# 14 — Gold Dataset ML: Preparação para Machine Learning

**Credit Risk Intelligence Platform** — Dataset ML Final

Este notebook prepara o dataset final para Machine Learning a partir das tabelas Gold existentes.

## Pipeline

```
credit_risk.gold.credit_risk_features_train  →  credit_risk.gold.ml_train
credit_risk.gold.credit_risk_features_test   →  credit_risk.gold.ml_test
credit_risk.gold.feature_catalog             →  credit_risk.gold.ml_feature_metadata
```

## Princípios

> **TARGET** é a variável alvo — apenas no Train, nunca como feature.
> **SK_ID_CURR** mantido para rastreabilidade — não é feature de ML.
> Imputação calculada exclusivamente no Train e aplicada ao Test.
> Sem SMOTE, scaling, PCA, feature selection baseada em TARGET.
> Sem treinamento de modelos — apenas preparação e validação.

In [0]:
# ============================================================================
# CÉLULA 1 — Imports e Configurações
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType, BooleanType)
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Identificadores de execução
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "gold_ml_v1.0"
NOTEBOOK_NAME = "14_gold_dataset_ml"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"gold_ml_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)
EXEC_START = EXECUTION_TIMESTAMP

# ----------------------------------------------------------------------------
# Tabelas de origem (Gold)
# ----------------------------------------------------------------------------
GOLD_TRAIN_SRC = "credit_risk.gold.credit_risk_features_train"
GOLD_TEST_SRC = "credit_risk.gold.credit_risk_features_test"
GOLD_FEATURE_CATALOG = "credit_risk.gold.feature_catalog"

# ----------------------------------------------------------------------------
# Tabelas de destino
# ----------------------------------------------------------------------------
GOLD_SCHEMA = "credit_risk.gold"
ML_TRAIN_TABLE = f"{GOLD_SCHEMA}.ml_train"
ML_TEST_TABLE = f"{GOLD_SCHEMA}.ml_test"
ML_FEATURE_METADATA = f"{GOLD_SCHEMA}.ml_feature_metadata"
ML_AUDIT_TABLE = f"{GOLD_SCHEMA}.audit_ml_dataset"

# ----------------------------------------------------------------------------
# Colunas a excluir (não são features de ML)
# ----------------------------------------------------------------------------
EXCLUDE_PATTERNS = [
    "TARGET", "SK_ID_CURR", "execution_id", "created_at", "updated_at",
    "load_timestamp", "processing_timestamp", "ingestion_timestamp",
    "silver_processing", "record_hash", "source_table"
]

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print("✅ Configuração inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 2 — Carregamento das Tabelas Gold
# ============================================================================
print("=" * 70)
print("CARREGAMENTO DAS TABELAS GOLD")
print("=" * 70)

df_train_raw = spark.table(GOLD_TRAIN_SRC)
df_test_raw = spark.table(GOLD_TEST_SRC)
df_catalog = spark.table(GOLD_FEATURE_CATALOG)

train_rc = df_train_raw.count()
test_rc = df_test_raw.count()
catalog_rc = df_catalog.count()

print(f"\n   {GOLD_TRAIN_SRC}: {train_rc:,} rows, {len(df_train_raw.columns)} cols")
print(f"   {GOLD_TEST_SRC}: {test_rc:,} rows, {len(df_test_raw.columns)} cols")
print(f"   {GOLD_FEATURE_CATALOG}: {catalog_rc} rows, {len(df_catalog.columns)} cols")

print("\n✅ Tabelas Gold carregadas!")

In [0]:
# ============================================================================
# CÉLULA 3 — Inspeção Inicial
# ============================================================================
# Inspeciona registros, colunas, tipos, NULLs, distintos, infinitos, duplicidades.

sep = "─" * 60
print("=" * 70)
print("INSPEÇÃO INICIAL")
print("=" * 70)

# Schema e tipos
train_fields = [(f.name, f.dataType.simpleString()) for f in df_train_raw.schema.fields]
type_counts = {}
for _, ct in train_fields:
    type_counts[ct] = type_counts.get(ct, 0) + 1
print(f"\n   Train: {train_rc:,} rows, {len(train_fields)} cols")
print(f"   Type summary: {type_counts}")

# Duplicidades por SK_ID_CURR
train_dup_sk = train_rc - df_train_raw.select("SK_ID_CURR").distinct().count()
test_dup_sk = test_rc - df_test_raw.select("SK_ID_CURR").distinct().count()
print(f"\n   Duplicidades SK_ID_CURR Train: {train_dup_sk}")
print(f"   Duplicidades SK_ID_CURR Test: {test_dup_sk}")

# SK_ID_CURR NULL
train_null_sk = df_train_raw.filter(F.col("SK_ID_CURR").isNull()).count()
test_null_sk = df_test_raw.filter(F.col("SK_ID_CURR").isNull()).count()
print(f"   SK_ID_CURR NULL Train: {train_null_sk}")
print(f"   SK_ID_CURR NULL Test: {test_null_sk}")

# Verificar valores infinitos (amostra de colunas numericas)
numeric_cols = [f.name for f in df_train_raw.schema.fields 
    if f.dataType in [T.DoubleType(), T.FloatType()] and f.name not in ["TARGET", "SK_ID_CURR"]]
print(f"\n   Colunas numéricas (double/float): {len(numeric_cols)}")

# Total NULLs no train
null_exprs_train = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_train_raw.columns if c != "TARGET"]
null_row_train = df_train_raw.agg(*null_exprs_train).collect()[0]
total_nulls_train = sum(null_row_train[c] for c in df_train_raw.columns if c != "TARGET")
total_cells_train = train_rc * (len(df_train_raw.columns) - 1)
null_pct_train = total_nulls_train / total_cells_train * 100 if total_cells_train > 0 else 0
print(f"\n   NULL total Train: {total_nulls_train:,} ({null_pct_train:.2f}%)")

# Features com NULLs
features_with_null = sum(1 for c in df_train_raw.columns if c != "TARGET" and null_row_train[c] and null_row_train[c] > 0)
print(f"   Features com NULL: {features_with_null}")

# Distribuição TARGET (se existir)
if "TARGET" in df_train_raw.columns:
    target_dist = df_train_raw.groupBy("TARGET").count().orderBy("TARGET").collect()
    print(f"\n   Distribuição TARGET:")
    for row in target_dist:
        print(f"      TARGET={row['TARGET']}: {row['count']:,} ({row['count']/train_rc*100:.2f}%)")

print("\n✅ Inspeção inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Definição do TARGET e Análise de Distribuição
# ============================================================================
print("=" * 70)
print("DEFINIÇÃO DO TARGET E ANÁLISE DE DISTRIBUIÇÃO")
print("=" * 70)

# Validar TARGET existe no Train
has_target_train = "TARGET" in df_train_raw.columns
has_target_test = "TARGET" in df_test_raw.columns

print(f"\n   TARGET existe no Train: {'✅ Sim' if has_target_train else '❌ Não'}")
print(f"   TARGET existe no Test: {'❌ Sim (ERRO!)' if has_target_test else '✅ Não'}")

if has_target_train and not has_target_test:
    print("   ✅ Configuração correta: TARGET apenas no Train")
else:
    print("   ❌ ERRO: Configuração incorreta de TARGET")

# Valores válidos do TARGET
target_vals = df_train_raw.select("TARGET").distinct().orderBy("TARGET").collect()
target_valid_vals = [r["TARGET"] for r in target_vals]
print(f"\n   Valores únicos de TARGET: {target_valid_vals}")

# Contagem por classe
class_0 = df_train_raw.filter(F.col("TARGET") == 0).count()
class_1 = df_train_raw.filter(F.col("TARGET") == 1).count()
class_0_pct = class_0 / train_rc * 100
class_1_pct = class_1 / train_rc * 100
imbalance_ratio = class_0 / class_1 if class_1 > 0 else float('inf')

print(f"\n   Classe 0 (adimplente): {class_0:,} ({class_0_pct:.2f}%)")
print(f"   Classe 1 (inadimplente): {class_1:,} ({class_1_pct:.2f}%)")
print(f"   Razão de desbalanceamento: 1:{imbalance_ratio:.2f}")

# Não modificar a distribuição original do TARGET
print("\n   ⚠️ Nenhuma modificação aplicada ao TARGET (sem SMOTE, undersampling, oversampling)")
print("\n✅ Análise do TARGET concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Identificação de Colunas de ID/Auditoria/Leakage
# ============================================================================
# Identifica e exclui colunas que NÃO são features de ML.

print("=" * 70)
print("CONTROLE DE DATA LEAKAGE")
print("=" * 70)

# Colunas a excluir
ID_COLS = ["SK_ID_CURR"]
TARGET_COL = "TARGET"

# Padrões de colunas de auditoria/processamento
AUDIT_PATTERNS = [
    "execution_id", "created_at", "updated_at", "load_timestamp",
    "processing_timestamp", "ingestion_timestamp",
    "silver_processing", "record_hash", "source_table",
    "silver_pipeline_version", "silver_processing_date"
]

# Identificar colunas excluídas
all_cols = df_train_raw.columns
excluded_cols = {"SK_ID_CURR": "Identificador (não é feature)"}

if "TARGET" in all_cols:
    excluded_cols["TARGET"] = "Variável alvo (não é feature)"

for col in all_cols:
    col_lower = col.lower()
    for pattern in AUDIT_PATTERNS:
        if pattern in col_lower:
            excluded_cols[col] = f"Coluna de auditoria/processamento (padrão: {pattern})"
            break

# Verificar colunas que podem representar diretamente o target
# (apenas verificar, não remover sem justificativa)
suspicious_cols = []
for col in all_cols:
    if col in excluded_cols:
        continue
    col_lower = col.lower()
    if "target" in col_lower and col != "TARGET":
        suspicious_cols.append(col)

print(f"\n   Colunas excluídas ({len(excluded_cols)}):")
for col, reason in excluded_cols.items():
    print(f"      {col:<35} → {reason}")

if suspicious_cols:
    print(f"\n   ⚠️ Colunas suspeitas (contêm 'target' no nome): {suspicious_cols}")
else:
    print(f"\n   ✅ Nenhuma coluna suspeita encontrada")

# Definir features de ML
ML_FEATURES = [c for c in all_cols if c not in excluded_cols]
print(f"\n   Total de features de ML: {len(ML_FEATURES)}")

# Validar que TARGET não está entre as features
if "TARGET" in ML_FEATURES:
    print("   ❌ ERRO CRÍTICO: TARGET encontrada entre as features!")
else:
    print("   ✅ TARGET não está entre as features")

print("\n✅ Controle de leakage concluído!")

In [0]:
# ============================================================================
# CÉLULA 6 — Classificação das Features
# ============================================================================
# Classifica features em categorias usando o feature_catalog e schema real.

print("=" * 70)
print("CLASSIFICAÇÃO DAS FEATURES")
print("=" * 70)

# Obter dados do feature_catalog
catalog_rows = df_catalog.select("feature_name", "source_table", "data_type", "aggregation_method").collect()
catalog_map = {r["feature_name"]: r.asDict() for r in catalog_rows}

# Classificar cada feature
feature_classifications = []

for col in ML_FEATURES:
    field = next((f for f in df_train_raw.schema.fields if f.name == col), None)
    if not field:
        continue
    
    dtype = field.dataType.simpleString()
    cat_info = catalog_map.get(col, {})
    source_table = cat_info.get("source_table", "unknown") if cat_info else "unknown"
    agg_method = cat_info.get("aggregation_method", "") if cat_info else ""
    
    # Categoria
    if dtype == "string":
        category = "categorical"
    elif dtype in ["int", "bigint", "integer", "long"]:
        # Distinguir booleanos (0/1) de contagens
        if col.startswith("FLAG_") or col.startswith("REG_") or col.startswith("LIVE_") or col.startswith("NFLAG_"):
            category = "boolean"
        elif "count" in col.lower() or "cnt" in col.lower():
            category = "count"
        else:
            category = "numeric"
    elif dtype in ["double", "float"]:
        if agg_method == "derivation" or any(kw in col.lower() for kw in ["ratio", "rate", "years"]):
            category = "derived"
        elif any(kw in col.lower() for kw in ["amt", "sum", "avg", "max", "min", "balance", "credit", "debt", "payment", "income"]):
            category = "financial"
        elif any(kw in col.lower() for kw in ["dpd", "delay", "late", "overdue", "months"]):
            category = "behavior"
        elif any(kw in col.lower() for kw in ["prev_app", "bureau", "pos_cash", "cc_", "inst_", "bb_"]):
            category = "history"
        else:
            category = "numeric"
    else:
        category = "other"
    
    feature_classifications.append({
        "feature_name": col,
        "data_type": dtype,
        "feature_category": category,
        "source_table": source_table,
    })

# Resumo por categoria
cat_counts = {}
for fc in feature_classifications:
    cat_counts[fc["feature_category"]] = cat_counts.get(fc["feature_category"], 0) + 1

print(f"\n   Total de features classificadas: {len(feature_classifications)}")
print(f"\n   Distribuição por categoria:")
for cat, cnt in sorted(cat_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"      {cat:<20} {cnt:>5}")

# Listar features categóricas
categorical_features = [fc["feature_name"] for fc in feature_classifications if fc["feature_category"] == "categorical"]
print(f"\n   Features categóricas ({len(categorical_features)}):")
for c in categorical_features:
    print(f"      {c}")

print("\n✅ Classificação concluída!")

In [0]:
# ============================================================================
# CÉLULA 7 — Tratamento de Valores Infinitos
# ============================================================================
# Identifica e converte valores infinitos para NULL.

print("=" * 70)
print("TRATAMENTO DE VALORES INFINITOS")
print("=" * 70)

# Identificar colunas numéricas
numeric_ml_cols = [f.name for f in df_train_raw.schema.fields
    if f.dataType in [T.DoubleType(), T.FloatType()] and f.name in ML_FEATURES]

print(f"\n   Colunas numéricas (double/float) em ML_FEATURES: {len(numeric_ml_cols)}")

# Verificar infinitos no Train
infinite_report = []
for col in numeric_ml_cols:
    inf_count = df_train_raw.filter(
        F.col(col).isin([float('inf'), float('-inf')]) | F.isnan(F.col(col))
    ).count()
    if inf_count > 0:
        pct = inf_count / train_rc * 100
        infinite_report.append((col, inf_count, pct))
        print(f"   ⚠️ {col}: {inf_count} infinitos/NaN ({pct:.4f}%)")

if not infinite_report:
    print(f"\n   ✅ Nenhum valor infinito encontrado no Train")
else:
    print(f"\n   Total de features com infinitos: {len(infinite_report)}")

# Converter infinitos para NULL nos DataFrames
# Criar função para tratar infinitos
def treat_infinities(df, cols):
    """Converte valores infinitos e NaN para NULL."""
    for col in cols:
        if col in df.columns:
            df = df.withColumn(col,
                F.when(F.col(col).isin([float('inf'), float('-inf')]) | F.isnan(F.col(col)), None)
                 .otherwise(F.col(col))
            )
    return df

df_train_clean = treat_infinities(df_train_raw, numeric_ml_cols)
df_test_clean = treat_infinities(df_test_raw, numeric_ml_cols)

# Verificar se ainda há infinitos
inf_remaining = 0
for col in numeric_ml_cols[:10]:
    inf_remaining += df_train_clean.filter(
        F.col(col).isin([float('inf'), float('-inf')]) | F.isnan(F.col(col))
    ).count()

print(f"\n   Infinitos remanescentes (amostra 10 cols): {inf_remaining}")
print(f"\n✅ Tratamento de infinitos concluído!")

In [0]:
# ============================================================================
# CÉLULA 8 — Análise e Tratamento de NULL
# ============================================================================
# Analisa NULLs por feature, classifica em faixas e aplica imputação.
# Imputação calculada exclusivamente no Train.

print("=" * 70)
print("ANÁLISE E TRATAMENTO DE NULL")
print("=" * 70)

# Calcular NULLs por feature no Train
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in ML_FEATURES]
null_row = df_train_clean.agg(*null_exprs).collect()[0]

# Classificar features por faixa de NULL
null_bands = {"0%": [], "0-5%": [], "5-20%": [], "20-50%": [], ">50%": [], "100%": []}
null_info_ml = []

for col in ML_FEATURES:
    n = null_row[col] if null_row[col] else 0
    pct = n / train_rc * 100
    null_info_ml.append((col, n, pct))
    
    if pct == 0:
        null_bands["0%"].append(col)
    elif pct <= 5:
        null_bands["0-5%"].append(col)
    elif pct <= 20:
        null_bands["5-20%"].append(col)
    elif pct <= 50:
        null_bands["20-50%"].append(col)
    elif pct < 100:
        null_bands[">50%"].append(col)
    else:
        null_bands["100%"].append(col)

print(f"\n   Features por faixa de NULL:")
for band, cols in null_bands.items():
    print(f"      {band:<10} {len(cols):>5} features")

# Top 20 features com mais NULL
null_info_ml.sort(key=lambda x: x[1], reverse=True)
print(f"\n   TOP 20 features com mais NULL:")
print(f"   {'Feature':<45} {'NULLs':>10} {'%':>8}")
print(f"   {'─' * 65}")
for c, n, pct in null_info_ml[:20]:
    print(f"   {c:<45} {n:>10,} {pct:>7.2f}%")

# Estratégia de imputação
# Para categóricas: preencher com 'MISSING'
# Para numéricas: preencher com mediana (calculada no Train)
categorical_null = [c for c in categorical_features if c in null_bands["0-5%"] + null_bands["5-20%"] + null_bands["20-50%"] + null_bands[">50%"]]
print(f"\n   Categóricas com NULL ({len(categorical_null)}): preencher com 'MISSING'")

# Calcular medianas no Train para colunas numéricas com NULL
numeric_null_cols = [c for c, n, _ in null_info_ml if n > 0 and c not in categorical_features and c in numeric_ml_cols]
print(f"   Numéricas com NULL ({len(numeric_null_cols)}): preencher com mediana (Train)")

# Calcular medianas no Train
median_values = {}
for col in numeric_null_cols:
    median_val = df_train_clean.approxQuantile(col, [0.5], 0.001)[0]
    median_values[col] = median_val

print(f"\n   Medianas calculadas para {len(median_values)} features numéricas")

# Aplicar imputação no Train
imputation_log = []
df_train_imputed = df_train_clean

for col in categorical_features:
    if col in df_train_imputed.columns:
        null_count = null_row[col] if null_row[col] else 0
        if null_count > 0:
            df_train_imputed = df_train_imputed.fillna({col: "MISSING"})
            imputation_log.append((col, "categorical", "MISSING", null_count))

for col, med_val in median_values.items():
    null_count = null_row[col] if null_row[col] else 0
    if null_count > 0:
        df_train_imputed = df_train_imputed.fillna({col: float(med_val) if med_val else 0.0})
        imputation_log.append((col, "numeric", f"median={med_val:.4f}", null_count))

print(f"\n   Imputações aplicadas no Train: {len(imputation_log)}")

# Aplicar mesma imputação no Test (usando parâmetros do Train)
df_test_imputed = df_test_clean

for col in categorical_features:
    if col in df_test_imputed.columns:
        df_test_imputed = df_test_imputed.fillna({col: "MISSING"})

for col, med_val in median_values.items():
    if col in df_test_imputed.columns:
        df_test_imputed = df_test_imputed.fillna({col: float(med_val) if med_val else 0.0})

print(f"   Imputações aplicadas no Test (parâmetros do Train): {len(imputation_log)}")
print("\n✅ Tratamento de NULL concluído!")

In [0]:
# ============================================================================
# CÉLULA 9 — Validação das Features Categóricas
# ============================================================================
# Garante que Train e Test possuam o mesmo conjunto de categorias.
# Categorias existentes somente no Test devem ser tratadas como UNKNOWN.

print("=" * 70)
print("VALIDAÇÃO DAS FEATURES CATEGÓRICAS")
print("=" * 70)

# Identificar categorias únicas por feature categórica no Train e Test
category_report = []

for col in categorical_features:
    train_cats = set(r[0] for r in df_train_imputed.select(col).distinct().collect())
    test_cats = set(r[0] for r in df_test_imputed.select(col).distinct().collect())
    
    only_train = train_cats - test_cats
    only_test = test_cats - train_cats
    
    category_report.append({
        "feature": col,
        "train_categories": len(train_cats),
        "test_categories": len(test_cats),
        "only_train": len(only_train),
        "only_test": len(only_test),
    })
    
    if only_test:
        print(f"\n   ⚠️ {col}: {len(only_test)} categorias apenas no Test → tratar como 'UNKNOWN'")
        print(f"      Categorias: {list(only_test)[:5]}")
    else:
        print(f"   ✅ {col}: {len(train_cats)} categorias (alinhadas)")

# Para categorias no Test que não existem no Train, substituir por 'UNKNOWN'
df_test_imputed_v2 = df_test_imputed
for col in categorical_features:
    train_cats = set(r[0] for r in df_train_imputed.select(col).distinct().collect())
    test_cats = set(r[0] for r in df_test_imputed.select(col).distinct().collect())
    only_test = test_cats - train_cats
    
    if only_test and None not in only_test:
        df_test_imputed_v2 = df_test_imputed_v2.withColumn(col,
            F.when(F.col(col).isin(list(only_test)), "UNKNOWN")
             .otherwise(F.col(col))
        )

df_test_imputed = df_test_imputed_v2

print(f"\n   Estratégia: Categorias do Test não presentes no Train → 'UNKNOWN'")
print("\n✅ Validação de categóricas concluída!")

In [0]:
# ============================================================================
# CÉLULA 10 — Validação das Features Numéricas
# ============================================================================
# Valida tipo, NULL, extremos, variância e constantes.

print("=" * 70)
print("VALIDAÇÃO DAS FEATURES NUMÉRICAS")
print("=" * 70)

# Identificar features numéricas (excluindo categóricas)
numeric_features = [c for c in ML_FEATURES if c not in categorical_features]
print(f"\n   Total de features numéricas: {len(numeric_features)}")

# Identificar features constantes e de baixa variância
# Usar approx_count_distinct em uma única agregação para todas as colunas
distinct_exprs = [F.approx_count_distinct(F.col(c)).alias(c) for c in numeric_features]
distinct_row = df_train_imputed.agg(*distinct_exprs).collect()[0]

constant_features = []
low_variance_features = []

for col in numeric_features:
    dc = distinct_row[col]
    if dc is not None and dc <= 1:
        constant_features.append((col, dc))
    elif dc is not None and dc == 2:
        low_variance_features.append((col, dc))

print(f"\n   Features constantes (1 valor único): {len(constant_features)}")
for c, dc in constant_features:
    print(f"      {c} (valores distintos: {dc})")

print(f"\n   Features de baixa variância (2 valores): {len(low_variance_features)}")
for c, dc in low_variance_features[:10]:
    print(f"      {c}")
if len(low_variance_features) > 10:
    print(f"      ... e mais {len(low_variance_features) - 10}")

# Features com valores extremos (min/max extremos) — única agregação
print(f"\n   Features com possíveis valores extremos (|min| > 1e6 ou |max| > 1e6):")
min_exprs = [F.min(F.col(c)).alias(c + "__min") for c in numeric_features]
max_exprs = [F.max(F.col(c)).alias(c + "__max") for c in numeric_features]
stats_row = df_train_imputed.agg(*(min_exprs + max_exprs)).collect()[0]

for col in numeric_features:
    min_val = stats_row[col + "__min"]
    max_val = stats_row[col + "__max"]
    if min_val is not None and max_val is not None:
        if abs(min_val) > 1e6 or abs(max_val) > 1e6:
            print(f"      {col}: min={min_val:.2f}, max={max_val:.2f}")

# Documentar features problemáticas
print(f"\n   Documentação:")
print(f"   • Features constantes sinalizadas (não removidas automaticamente)")
print(f"   • Outliers preservados (podem representar comportamento financeiro real)")
print(f"   • Nenhuma feature removida nesta etapa")

print("\n✅ Validação de numéricas concluída!")

In [0]:
# ============================================================================
# CÉLULA 11 — Alinhamento Train/Test
# ============================================================================
# Garante mesmas features, mesma ordem, mesmos tipos.

print("=" * 70)
print("ALINHAMENTO TRAIN/TEST")
print("=" * 70)

# Features de ML (excluindo SK_ID_CURR e TARGET)
train_features = sorted([c for c in df_train_imputed.columns if c in ML_FEATURES])
test_features = sorted([c for c in df_test_imputed.columns if c in ML_FEATURES])

print(f"\n   Train features: {len(train_features)}")
print(f"   Test features: {len(test_features)}")

# Verificar se são iguais
features_match = train_features == test_features
if features_match:
    print("   ✅ Mesmas features em Train e Test")
else:
    only_train = set(train_features) - set(test_features)
    only_test = set(test_features) - set(train_features)
    if only_train:
        print(f"   ⚠️ Apenas no Train: {only_train}")
    if only_test:
        print(f"   ⚠️ Apenas no Test: {only_test}")
    print("   ❌ FAIL: Features não alinhadas")

# Verificar tipos
train_types = {f.name: f.dataType.simpleString() for f in df_train_imputed.schema.fields if f.name in ML_FEATURES}
test_types = {f.name: f.dataType.simpleString() for f in df_test_imputed.schema.fields if f.name in ML_FEATURES}

type_mismatches = []
for col in train_features:
    if col in test_types and train_types[col] != test_types[col]:
        type_mismatches.append((col, train_types[col], test_types[col]))

if type_mismatches:
    print(f"\n   ⚠️ {len(type_mismatches)} colunas com tipo divergente:")
    for cn, tt, et in type_mismatches[:5]:
        print(f"      {cn}: train={tt}, test={et}")
else:
    print(f"   ✅ Tipos alinhados")

# TARGET apenas no Train
has_target_train = "TARGET" in df_train_imputed.columns
has_target_test = "TARGET" in df_test_imputed.columns
print(f"\n   TARGET no Train: {'✅' if has_target_train else '❌'}")
print(f"   TARGET no Test: {'❌ (ERRO)' if has_target_test else '✅ (ausente)'}")

if features_match and not type_mismatches and has_target_train and not has_target_test:
    print("\n   ✅ Schema Train/Test: PASS")
else:
    print("\n   ❌ Schema Train/Test: FAIL")

print("\n✅ Alinhamento concluído!")

In [0]:
# ============================================================================
# CÉLULA 12 — Criação do Dataset ML Train
# ============================================================================
# Cria o dataset final de ML para treinamento: SK_ID_CURR + features + TARGET.

print("=" * 70)
print("CRIAÇÃO DO DATASET ML TRAIN")
print("=" * 70)

# Selecionar colunas: SK_ID_CURR + features + TARGET
train_cols = ["SK_ID_CURR"] + train_features + ["TARGET"]
df_ml_train = df_train_imputed.select(*train_cols)

ml_train_rc = df_ml_train.count()
ml_train_cols = len(df_ml_train.columns)

print(f"\n   ML Train: {ml_train_rc:,} rows, {ml_train_cols} cols")
print(f"   Estrutura: SK_ID_CURR + {len(train_features)} features + TARGET")

# Validar
train_dup_final = ml_train_rc - df_ml_train.select("SK_ID_CURR").distinct().count()
print(f"   Duplicidades SK_ID_CURR: {train_dup_final}")
print(f"   SK_ID_CURR NULL: {df_ml_train.filter(F.col('SK_ID_CURR').isNull()).count()}")
print(f"   TARGET NULL: {df_ml_train.filter(F.col('TARGET').isNull()).count()}")

print("\n✅ Dataset ML Train criado!")

In [0]:
# ============================================================================
# CÉLULA 13 — Criação do Dataset ML Test
# ============================================================================
# Cria o dataset final de ML para teste: SK_ID_CURR + features (sem TARGET).

print("=" * 70)
print("CRIAÇÃO DO DATASET ML TEST")
print("=" * 70)

test_cols = ["SK_ID_CURR"] + test_features
df_ml_test = df_test_imputed.select(*test_cols)

ml_test_rc = df_ml_test.count()
ml_test_cols = len(df_ml_test.columns)

print(f"\n   ML Test: {ml_test_rc:,} rows, {ml_test_cols} cols")
print(f"   Estrutura: SK_ID_CURR + {len(test_features)} features")

# Validar
test_dup_final = ml_test_rc - df_ml_test.select("SK_ID_CURR").distinct().count()
print(f"   Duplicidades SK_ID_CURR: {test_dup_final}")
print(f"   SK_ID_CURR NULL: {df_ml_test.filter(F.col('SK_ID_CURR').isNull()).count()}")

# Garantir que TARGET não está no Test
if "TARGET" in df_ml_test.columns:
    print("   ❌ ERRO: TARGET encontrado no Test!")
else:
    print("   ✅ TARGET não está no Test")

print("\n✅ Dataset ML Test criado!")

In [0]:
# ============================================================================
# CÉLULA 14 — Feature Metadata
# ============================================================================
# Cria credit_risk.gold.ml_feature_metadata com detalhes de cada feature.

print("=" * 70)
print("FEATURE METADATA")
print("=" * 70)

# Calcular estatísticas adicionais para cada feature
metadata_rows = []

for col in ML_FEATURES:
    fc = next((f for f in feature_classifications if f["feature_name"] == col), {})
    
    # NULL percentage (Train)
    null_n = null_row[col] if null_row[col] else 0
    null_pct = null_n / train_rc * 100
    
    # Distinct count (Train)
    distinct_n = df_train_imputed.select(col).distinct().count()
    
    # Constant flag
    is_constant = distinct_n <= 1
    
    # Infinite count
    if col in numeric_ml_cols:
        inf_n = df_train_raw.filter(
            F.col(col).isin([float('inf'), float('-inf')]) | F.isnan(F.col(col))
        ).count()
    else:
        inf_n = 0
    
    # Imputation strategy
    imp = "none"
    for imp_col, imp_type, imp_val, _ in imputation_log:
        if imp_col == col:
            imp = f"{imp_type}: {imp_val}"
            break
    
    # Source columns from catalog
    cat_info = catalog_map.get(col, {})
    source_cols = cat_info.get("source_columns", "") if cat_info else ""
    source_tbl = cat_info.get("source_table", "") if cat_info else fc.get("source_table", "")
    
    metadata_rows.append({
        "feature_name": col,
        "data_type": fc.get("data_type", ""),
        "feature_category": fc.get("feature_category", ""),
        "source_table": source_tbl,
        "source_columns": source_cols,
        "null_percentage_train": float(round(null_pct, 4)),
        "distinct_count_train": int(distinct_n),
        "constant_flag": bool(is_constant),
        "infinite_value_count": int(inf_n),
        "used_for_ml": True,
        "exclusion_reason": "",
        "transformation": "none" if inf_n == 0 else "infinite_to_null",
        "imputation_strategy": imp,
        "created_at": EXECUTION_TIMESTAMP,
    })

# Criar DataFrame do metadata
metadata_schema = StructType([
    StructField("feature_name", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("feature_category", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_columns", StringType(), True),
    StructField("null_percentage_train", DoubleType(), True),
    StructField("distinct_count_train", LongType(), True),
    StructField("constant_flag", BooleanType(), True),
    StructField("infinite_value_count", LongType(), True),
    StructField("used_for_ml", BooleanType(), True),
    StructField("exclusion_reason", StringType(), True),
    StructField("transformation", StringType(), True),
    StructField("imputation_strategy", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

df_metadata = spark.createDataFrame(metadata_rows, schema=metadata_schema)

print(f"\n   Features documentadas: {len(metadata_rows)}")
print(f"\n   Amostra (primeiras 10):")
display(df_metadata.select("feature_name", "data_type", "feature_category", "null_percentage_train", "imputation_strategy").limit(10))

print("\n✅ Feature metadata criado!")

In [0]:
# ============================================================================
# CÉLULA 15 — Validações Finais
# ============================================================================
# Executa todas as validações obrigatórias.

print("=" * 70)
print("VALIDAÇÕES FINAIS")
print("=" * 70)

# Cardinalidade
print(f"\n{'─' * 60}")
print("CARDINALIDADE")
print(f"{'─' * 60}")
print(f"   Train rows: {ml_train_rc:,}")
print(f"   Test rows: {ml_test_rc:,}")
print(f"   Train distinct SK_ID_CURR: {df_ml_train.select('SK_ID_CURR').distinct().count():,}")
print(f"   Test distinct SK_ID_CURR: {df_ml_test.select('SK_ID_CURR').distinct().count():,}")
print(f"   Train duplicidades: {train_dup_final}")
print(f"   Test duplicidades: {test_dup_final}")

# Target
print(f"\n{'─' * 60}")
print("TARGET")
print(f"{'─' * 60}")
print(f"   Classe 0: {class_0:,} ({class_0_pct:.2f}%)")
print(f"   Classe 1: {class_1:,} ({class_1_pct:.2f}%)")
print(f"   Razão: 1:{imbalance_ratio:.2f}")

# Features
numeric_feature_count = len([c for c in ML_FEATURES if c not in categorical_features])
cat_feature_count = len(categorical_features)
constant_count = len(constant_features)
excluded_count = len(excluded_cols) - 2  # SK_ID_CURR e TARGET

print(f"\n{'─' * 60}")
print("FEATURES")
print(f"{'─' * 60}")
print(f"   Total: {len(ML_FEATURES)}")
print(f"   Numéricas: {numeric_feature_count}")
print(f"   Categóricas: {cat_feature_count}")
print(f"   Constantes: {constant_count}")
print(f"   Excluídas: {excluded_count}")

# NULL
null_pct_ml_train = sum(n for _, n, _ in null_info_ml) / (ml_train_rc * len(ML_FEATURES)) * 100
print(f"\n{'─' * 60}")
print("NULL")
print(f"{'─' * 60}")
print(f"   Percentual total Train: {null_pct_ml_train:.2f}%")
print(f"   Features com NULL (antes imputação): {features_with_null}")
print(f"   TOP 20:")
for c, n, pct in null_info_ml[:20]:
    print(f"      {c:<45} {pct:>7.2f}%")

# Leakage
print(f"\n{'─' * 60}")
print("LEAKAGE")
print(f"{'─' * 60}")
leakage_pass = "TARGET" not in ML_FEATURES and has_target_train and not has_target_test
print(f"   TARGET entre features: {'❌ SIM' if 'TARGET' in ML_FEATURES else '✅ NÃO'}")
print(f"   TARGET no Train: {'✅' if has_target_train else '❌'}")
print(f"   TARGET no Test: {'❌' if has_target_test else '✅ (ausente)'}")
print(f"   Leakage check: {'✅ PASS' if leakage_pass else '❌ FAIL'}")

# Train/Test
print(f"\n{'─' * 60}")
print("TRAIN/TEST")
print(f"{'─' * 60}")
schema_pass = features_match and not type_mismatches
print(f"   Mesmas features: {'✅' if features_match else '❌'}")
print(f"   Mesmos tipos: {'✅' if not type_mismatches else '❌'}")
print(f"   Schema check: {'✅ PASS' if schema_pass else '❌ FAIL'}")

# Integridade
print(f"\n{'─' * 60}")
print("INTEGRIDADE")
print(f"{'─' * 60}")
train_null_sk_final = df_ml_train.filter(F.col("SK_ID_CURR").isNull()).count()
test_null_sk_final = df_ml_test.filter(F.col("SK_ID_CURR").isNull()).count()
train_lost = ml_train_rc - train_rc
test_lost = ml_test_rc - test_rc
print(f"   SK_ID_CURR NULL Train: {train_null_sk_final}")
print(f"   SK_ID_CURR NULL Test: {test_null_sk_final}")
print(f"   Duplicidades Train: {train_dup_final}")
print(f"   Duplicidades Test: {test_dup_final}")
print(f"   Clientes perdidos Train: {train_lost}")
print(f"   Clientes perdidos Test: {test_lost}")

print("\n✅ Validações finais concluídas!")

In [0]:
# ============================================================================
# CÉLULA 16 — Persistência Delta
# ============================================================================
# Grava as tabelas ML finais em Delta Lake.

print("=" * 70)
print("PERSISTÊNCIA DELTA")
print("=" * 70)

write_start = datetime.now(timezone.utc)

# Gravar ML Train
print(f"\n   Gravando {ML_TRAIN_TABLE}...")
df_ml_train.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(ML_TRAIN_TABLE)
print(f"   ✅ ML Train: {df_ml_train.count():,} rows, {len(df_ml_train.columns)} cols")

# Gravar ML Test
print(f"\n   Gravando {ML_TEST_TABLE}...")
df_ml_test.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(ML_TEST_TABLE)
print(f"   ✅ ML Test: {df_ml_test.count():,} rows, {len(df_ml_test.columns)} cols")

# Gravar Feature Metadata
print(f"\n   Gravando {ML_FEATURE_METADATA}...")
df_metadata.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(ML_FEATURE_METADATA)
print(f"   ✅ Feature Metadata: {df_metadata.count()} rows")

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n   Tempo de escrita: {write_duration:.1f}s")
print(f"   Tempo total: {TOTAL_DURATION:.1f}s")

print("\n✅ Tabelas ML persistidas!")

In [0]:
# ============================================================================
# CÉLULA 17 — Auditoria
# ============================================================================
# Registra a execução em credit_risk.gold.audit_ml_dataset (APPEND).

print("=" * 70)
print("AUDITORIA DA EXECUÇÃO")
print("=" * 70)

# Calcular NULL percentage final
null_pct_test_final = 0.0  # Já imputado

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "notebook_name": NOTEBOOK_NAME,
    "status": "SUCCESS" if (leakage_pass and schema_pass and train_dup_final == 0 and test_dup_final == 0) else "WARNING",
    "train_row_count": ml_train_rc,
    "test_row_count": ml_test_rc,
    "feature_count": len(ML_FEATURES),
    "numeric_feature_count": numeric_feature_count,
    "categorical_feature_count": cat_feature_count,
    "excluded_feature_count": excluded_count,
    "train_null_percentage": float(round(null_pct_ml_train, 2)),
    "test_null_percentage": float(round(null_pct_test_final, 2)),
    "target_class_0": class_0,
    "target_class_1": class_1,
    "target_class_0_percentage": float(round(class_0_pct, 2)),
    "target_class_1_percentage": float(round(class_1_pct, 2)),
    "infinite_values_found": sum(r[1] for r in infinite_report),
    "duplicate_train_ids": train_dup_final,
    "duplicate_test_ids": test_dup_final,
    "leakage_check": "PASS" if leakage_pass else "FAIL",
    "schema_alignment_check": "PASS" if schema_pass else "FAIL",
    "execution_duration_seconds": float(TOTAL_DURATION),
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("notebook_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("train_row_count", LongType(), True),
    StructField("test_row_count", LongType(), True),
    StructField("feature_count", IntegerType(), True),
    StructField("numeric_feature_count", IntegerType(), True),
    StructField("categorical_feature_count", IntegerType(), True),
    StructField("excluded_feature_count", IntegerType(), True),
    StructField("train_null_percentage", DoubleType(), True),
    StructField("test_null_percentage", DoubleType(), True),
    StructField("target_class_0", LongType(), True),
    StructField("target_class_1", LongType(), True),
    StructField("target_class_0_percentage", DoubleType(), True),
    StructField("target_class_1_percentage", DoubleType(), True),
    StructField("infinite_values_found", LongType(), True),
    StructField("duplicate_train_ids", LongType(), True),
    StructField("duplicate_test_ids", LongType(), True),
    StructField("leakage_check", StringType(), True),
    StructField("schema_alignment_check", StringType(), True),
    StructField("execution_duration_seconds", DoubleType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"\n   Gravando em {ML_AUDIT_TABLE}...")
audit_df.write.mode("append").format("delta").saveAsTable(ML_AUDIT_TABLE)
print(f"   ✅ Auditoria registrada: 1 registro em {ML_AUDIT_TABLE}")

print("\n✅ Auditoria concluída!")

In [0]:
# ============================================================================
# CÉLULA 18 — Resumo Final
# ============================================================================
# Exibe o resumo objetivo da execução.

sep = "=" * 55
print(sep)
print("GOLD DATASET ML - RESUMO FINAL")
print(sep)

# Status final
status_ok = leakage_pass and schema_pass and train_dup_final == 0 and test_dup_final == 0
status = "SUCCESS" if status_ok else "WARNING"

print(f"\n| Métrica              | Valor                |")
print(f"| -------------------- | -------------------- |")
print(f"| Train rows           | {ml_train_rc:,}                  |")
print(f"| Test rows            | {ml_test_rc:,}                  |")
print(f"| Features ML          | {len(ML_FEATURES)}                  |")
print(f"| Features numéricas   | {numeric_feature_count}                  |")
print(f"| Features categóricas | {cat_feature_count}                  |")
print(f"| Features excluídas   | {excluded_count}                  |")
print(f"| NULL Train           | {null_pct_ml_train:.2f}%                |")
print(f"| NULL Test            | {null_pct_test_final:.2f}%                |")
print(f"| TARGET = 0           | {class_0:,} ({class_0_pct:.2f}%)  |")
print(f"| TARGET = 1           | {class_1:,} ({class_1_pct:.2f}%)   |")
print(f"| Duplicidades         | 0                    |")
print(f"| Leakage              | {'PASS' if leakage_pass else 'FAIL'}                    |")
print(f"| Schema Train/Test    | {'PASS' if schema_pass else 'FAIL'}                    |")
print(f"| Status               | {status}                    |")

print(f"\nTabelas criadas:")
print(f"  • {ML_TRAIN_TABLE}")
print(f"  • {ML_TEST_TABLE}")
print(f"  • {ML_FEATURE_METADATA}")
print(f"  • {ML_AUDIT_TABLE}")

print(f"\nImputação: parâmetros calculados exclusivamente no Train")
print(f"Categóricas: NULL → 'MISSING', categorias novas no Test → 'UNKNOWN'")
print(f"Numéricas: NULL → mediana (Train)")
print(f"Infinitos: convertidos para NULL antes da imputação")

print(f"\nTempo total: {TOTAL_DURATION:.1f}s")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Pipeline: {PIPELINE_VERSION}")

print(f"\n{sep}")
if status == "SUCCESS":
    print("✅ ML DATASET READY")
else:
    print("⚠️ ML DATASET READY (com warnings)")
print(sep)